In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr


In [ ]:
stadium = pd.read_csv('data/processed/clean_stadium.csv')
fanbase = pd.read_csv('data/processed/fanbase_clean.csv')
merch = pd.read_csv('data/processed/cleaned_merch.csv')
fan_merch_merged = pd.read_csv('data/processed/merch_fanbase_merged.csv')

## Calculate Fan Life Time Value by Age, Region, Season Pass demographics

In [ ]:
fan_ltv = fan_merch_merged.groupby('Member_ID').agg({
    'Unit_Price': 'sum', 
    'Barcode': 'count',
    'Games_Attended': 'first',   
    'Seasonal_Pass': 'first',  
    'Customer_Age_Group': 'first',
    'Customer_Region': 'first'
}).rename(columns={
    'Unit_Price': 'Total_Merch_Spend',
    'Barcode': 'Purchase_Count'
}).reset_index()


In [ ]:

ticket_sources = ['Upper Bowl', 'Lower Bowl', 'Season', 'Premium']
total_ticket_revenue = stadium[stadium['Source'].isin(ticket_sources)]['Revenue'].sum()
total_attendance = fan_merch_merged['Games_Attended'].sum() 

avg_ticket_price = total_ticket_revenue / total_attendance

print(f"Estimated average ticket price: ${avg_ticket_price:.2f}")
print(f"Total ticket revenue in dataset: ${total_ticket_revenue:,.2f}")
print(f"Total attendance in dataset: {total_attendance:,}")

In [ ]:
fan_ltv['Ticket_Revenue'] = fan_ltv['Games_Attended'] * avg_ticket_price

fan_ltv['Merchandise_LTV'] = fan_ltv['Total_Merch_Spend']
fan_ltv['Total_LTV'] = fan_ltv['Ticket_Revenue'] + fan_ltv['Merchandise_LTV']

In [ ]:
fan_ltv

In [ ]:
# Segment by Age Group
ltv_by_age = fan_ltv.groupby('Customer_Age_Group').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Member_ID': 'count'
}).round(2)

# Segment by Region
ltv_by_region = fan_ltv.groupby('Customer_Region').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Member_ID': 'count'
}).round(2)

# Segment by Season Pass Status
ltv_by_pass = fan_ltv.groupby('Seasonal_Pass').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Games_Attended': 'mean',
    'Member_ID': 'count'
}).round(2)


In [ ]:
ltv_by_age

In [ ]:
ltv_by_pass

In [ ]:
ltv_by_region

We find no correlation between number of games attending and money spent on merchendise

In [ ]:
# Calculate correlation between games attended and merchandise spending
correlation_data = fan_ltv[['Games_Attended', 'Merchandise_LTV']].copy()

# Remove any rows with missing values
correlation_data = correlation_data.dropna()

# Calculate correlation coefficients
pearson_corr, pearson_p = pearsonr(correlation_data['Games_Attended'], 
                                     correlation_data['Merchandise_LTV'])
spearman_corr, spearman_p = spearmanr(correlation_data['Games_Attended'], 
                                        correlation_data['Merchandise_LTV'])

print(f"Pearson Correlation: {pearson_corr:.3f} (p-value: {pearson_p:.4f})")
print(f"Spearman Correlation: {spearman_corr:.3f} (p-value: {spearman_p:.4f})")

# Create attendance bins 
fan_ltv['Attendance_Bin'] = pd.cut(fan_ltv['Games_Attended'], 
                                     bins=[0, 3, 8, 13, 20],
                                     labels=['Low (0-3)', 'Medium (4-8)', 
                                            'High (9-13)', 'Very High (14+)'])

# Compare merchandise spending across attendance bins
merch_by_attendance = fan_ltv.groupby('Attendance_Bin').agg({
    'Merchandise_LTV': ['mean', 'median', 'sum'],
    'Member_ID': 'count',
    'Games_Attended': 'mean'
}).round(2)

merch_by_attendance


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Create feature set for clustering
clustering_features = fan_ltv[['Games_Attended', 'Merchandise_LTV', 'Seasonal_Pass', 'Member_ID']].copy()

# Add additional behavioral features
# Purchase frequency
purchase_freq = fan_merch_merged.groupby('Member_ID').agg({
    'Barcode': 'count',  # Number of transactions
    'Item_Category': 'nunique',  # Variety of items purchased
    'Promotion': 'mean',  # Proportion of promotional purchases
    'Channel': lambda x: (x == 'Online').sum() / len(x)  # Online purchase ratio
}).rename(columns={
    'Barcode': 'Purchase_Count',
    'Item_Category': 'Item_Variety',
    'Promotion': 'Promo_Rate',
    'Channel': 'Online_Rate'
}).reset_index()



In [ ]:
clustering_features

In [ ]:
# Merge back to clustering dataset
clustering_data = clustering_features.merge(purchase_freq, on='Member_ID', how='left')
clustering_data = clustering_data.fillna(0)  # Fill NaN for members with no purchases

# Convert boolean to numeric
clustering_data['Seasonal_Pass'] = clustering_data['Seasonal_Pass'].astype(int)

print("Clustering Features:")
print(clustering_data.describe())

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
import random

# Standardize features (important for K-means)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(clustering_data)



In [ ]:
random.seed(123)

sample_size = min(10000, len(features_scaled)) 
sample_indices = np.random.choice(len(features_scaled), sample_size, replace=False)
features_sample = features_scaled[sample_indices]

# Run optimization on sample
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_sample)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(features_sample, kmeans.labels_))

# Then use optimal K on full dataset
optimal_k = K_range[np.argmax(silhouette_scores)]
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
final_labels = final_kmeans.fit_predict(features_scaled)

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].grid(True)

# Silhouette score
axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('Number of Clusters')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by K')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("\nSilhouette Scores:")
for k, score in zip(K_range, silhouette_scores):
    print(f"K={k}: {score:.3f}")

In [ ]:

optimal_k = 4 #anything more than 4 resulted in identical clusters, despite silhouette scores and elbow

# Fit final model
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features_scaled)

# Add cluster labels back to original data
fan_ltv['Cluster'] = cluster_labels
clustering_data['Cluster'] = cluster_labels

In [ ]:
# Calculate cluster characteristics
cluster_profiles = fan_ltv.groupby('Cluster').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Games_Attended': 'mean',
    'Seasonal_Pass': 'mean',  # Proportion with season pass
    'Member_ID': 'count'
}).round(2)

cluster_profiles.columns = ['_'.join(col).strip() for col in cluster_profiles.columns.values]
print("\nCLUSTER PROFILES")
print(cluster_profiles)

# Add behavioral characteristics
behavior_profiles = clustering_data.groupby('Cluster').agg({
    'Purchase_Count': 'mean',
    'Item_Variety': 'mean',
    'Promo_Rate': 'mean',
    'Online_Rate': 'mean'
}).round(2)

print("\nBEHAVIORAL PROFILES")
print(behavior_profiles)

# Demographic breakdown by cluster
demo_profiles = fan_ltv.groupby(['Cluster', 'Customer_Age_Group']).size().unstack(fill_value=0)
print("\nAGE DISTRIBUTION BY CLUSTER")
print(demo_profiles)

region_profiles = fan_ltv.groupby(['Cluster', 'Customer_Region']).size().unstack(fill_value=0)
print("\n REGION DISTRIBUTION BY CLUSTER ")
print(region_profiles)

The below code for visualizations was written with the help of Microsoft Copilot

In [ ]:
# PCA for visualization
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features_scaled)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(features_pca[:, 0], features_pca[:, 1], 
                     c=cluster_labels, cmap='viridis', alpha=0.6, s=50)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Fan Clusters (PCA Visualization)')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

# Radar chart for cluster comparison
from math import pi

categories = ['Games_Attended', 'Merchandise_LTV', 'Purchase_Count', 
              'Item_Variety', 'Seasonal_Pass']

# Normalize values for radar chart (0-1 scale)
cluster_radar = clustering_data.groupby('Cluster')[categories].mean()
cluster_radar_norm = (cluster_radar - cluster_radar.min()) / (cluster_radar.max() - cluster_radar.min())

fig, axes = plt.subplots(1, optimal_k, figsize=(20, 5), subplot_kw=dict(projection='polar'))

for idx, cluster in enumerate(range(optimal_k)):
    ax = axes[idx] if optimal_k > 1 else axes
    
    values = cluster_radar_norm.iloc[cluster].values.tolist()
    values += values[:1]  # Complete the circle
    
    angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'Cluster {cluster}', size=12, weight='bold')
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Create interpretable cluster names based on characteristics
def name_cluster(row):
    """
    Function to assign meaningful names to clusters
    Adjust logic based on your actual cluster profiles
    """
    cluster = row['Cluster']
    games = row['Games_Attended']
    merch = row['Merchandise_LTV']
    season_pass = row['Seasonal_Pass']
    
    # Example logic - adjust based on your results
    if games > 10 and merch > 200 and season_pass == 1:
        return 'Super Fans'
    elif games > 10 and season_pass == 1:
        return 'Loyal Attendees'
    elif merch > 150:
        return 'Merchandise Enthusiasts'
    elif games < 5 and merch < 50:
        return 'Casual Fans'
    else:
        return 'Moderate Fans'


print("\n=== CLUSTER SUMMARY ===")
for cluster in range(optimal_k):
    cluster_data = fan_ltv[fan_ltv['Cluster'] == cluster]
    print(f"\nCluster {cluster}:")
    print(f"  Size: {len(cluster_data)} fans")
    print(f"  Avg LTV: ${cluster_data['Total_LTV'].mean():.2f}")
    print(f"  Avg Games: {cluster_data['Games_Attended'].mean():.1f}")
    print(f"  Avg Merch: ${cluster_data['Merchandise_LTV'].mean():.2f}")
    print(f"  Season Pass %: {cluster_data['Seasonal_Pass'].mean()*100:.1f}%")

Cluster 0: Merchandise Enthusiasts
- increase attendance - ticket discounts?

Cluster 1: Disengaged Fans

Cluster 2: Super Fans/VIPs
- exclusive deals and early access (important to retain)

Cluster 3: Loyal Attendees
- upsell merchendise